# Import Libraries 

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Data Load


In [2]:
df = pd.read_csv("bank_churn_cleaned.csv")

In [23]:
# 1) Identificar colunas numéricas e categóricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# 2) Output (contagens + nomes)
print(f"Total de colunas: {df.shape[1]}\n")

print(f"Colunas NUMÉRICAS ({len(numeric_cols)}):")
print(numeric_cols)

print("\n" + "-"*80 + "\n")

Total de colunas: 23

Colunas NUMÉRICAS (17):
['CLIENTNUM', 'Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'unknown_count', 'churn']

--------------------------------------------------------------------------------



In [24]:
if "churn" not in df.columns:
    # Ajuste se seu Attrition_Flag tiver nomes diferentes
    df["churn"] = (df["Attrition_Flag"] == "Attrited Customer").astype(int)

In [32]:
# 2) Dropar colunas que você não quer usar
df_model = df.drop(columns=["CLIENTNUM", "unknown_count"], errors="ignore")

In [33]:
# 3) Separar X e y
X = df_model.drop(columns=["churn"], errors="ignore")
y = df_model["churn"]


In [34]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"Categóricas ({len(categorical_cols)}): {categorical_cols}")


Numéricas (14): ['Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']
Categóricas (6): ['Attrition_Flag', 'Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


In [35]:
# 5) Pipelines de tratamento
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [36]:
# 6) Pré-processador final (pronto pra plugar no modelo)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)

# Exemplo: só para testar que transforma sem erro
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

print("Shape X_train antes:", X_train.shape)
print("Shape X_train depois:", X_train_transformed.shape)

Shape X_train antes: (8101, 20)
Shape X_train depois: (8101, 39)
